In [1]:
%env XLA_PYTHON_CLIENT_MEM_FRACTION=.25
%env JAX_LOG_COMPILES=1

env: XLA_PYTHON_CLIENT_MEM_FRACTION=.25
env: JAX_LOG_COMPILES=1


In [2]:
import jax
jax.config.update('jax_threefry_partitionable', True)
import netket as nk

import netket.experimental
from functools import partial

# from jax.config import config
# config.update("jax_enable_x64", False)
# del config

Finished tracing + transforming jit(convert_element_type) in 0.00028705596923828125 sec
Finished tracing + transforming jit(broadcast_in_dim) in 0.0002758502960205078 sec
Compiling broadcast_in_dim for with global shapes and types [ShapedArray(float64[])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module conversion jit(broadcast_in_dim) in 0.0014233589172363281 sec
Finished XLA compilation of jit(broadcast_in_dim) in 1.8684217929840088 sec
Finished tracing + transforming jit(broadcast_in_dim) in 0.0002753734588623047 sec
Compiling broadcast_in_dim for with global shapes and types [ShapedArray(float64[])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module conversion jit(broadcast_in_dim) in 0.0009641647338867188 sec
Finished XLA compilation of jit(broadcast_in_dim) in 0.011644363403320312 sec
Finished tracing + transforming jit(convert_element_type) in 0.00018215179443359375 sec
Finished tracing + transforming jit(convert_elem

In [3]:
jax.devices()

[gpu(id=0), gpu(id=1)]

In [4]:
L = 32

n_chains = 1024

g = nk.graph.Hypercube(length=L, n_dim=1, pbc=True)
hi = nk.hilbert.Spin(s=1 / 2, N=g.n_nodes)
ha = nk.operator.Ising(hilbert=hi, graph=g, h=1.0)
ma = nk.models.RBM(alpha=8, param_dtype=complex)
#ma = nk.models.GCNN(g, features=8, layers=4, param_dtype=complex, mode='fft')
sa = nk.sampler.MetropolisLocal(hi, n_chains=512)
op = nk.optimizer.Sgd(learning_rate=0.1)

In [5]:
sr = nk.optimizer.SR(diag_shift=0.01, qgt=nk.optimizer.qgt.QGTOnTheFly)
srp = nk.optimizer.SR(diag_shift=0.01, qgt=partial(nk.optimizer.qgt.QGTJacobianPyTree, holomorphic=True))

In [6]:
vs1 = nk.vqs.MCState(sa, ma, n_samples=8192, n_discard_per_chain=0)
sampler_state1 =  vs1.sampler_state

sharding = jax.sharding.PositionalSharding(jax.devices())
vs2 = nk.vqs.MCState(sa, ma, n_samples=8192, n_discard_per_chain=0)
sampler_state2 = vs2.sampler_state.replace(σ=jax.device_put(vs2.sampler_state.σ, sharding.reshape(-1, 1)))
vs2.sampler_state = sampler_state2
# 
# x = vs.samples
# p = vs.variables
# f = vs._apply_fun
# lowered = jax.jit(jax.vmap(f, in_axes=(None, 0))).lower(p, x)
# compiled = lowered.compile()
# ca = compiled.cost_analysis()
# intensity = ca[0]['flops'] / ca[0]['bytes accessed']
# print('Comp. intensity fwd pass:', intensity, 'flops/byte')
#

Finished tracing + transforming jit(convert_element_type) in 0.00017404556274414062 sec
Finished tracing + transforming <lambda> for pjit in 0.00034165382385253906 sec
Finished tracing + transforming _threefry_seed for pjit in 0.0016808509826660156 sec
Compiling _threefry_seed for with global shapes and types [ShapedArray(int64[])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module conversion jit(_threefry_seed) in 0.0021142959594726562 sec
Finished XLA compilation of jit(_threefry_seed) in 0.04655885696411133 sec
Finished tracing + transforming jit(broadcast_in_dim) in 0.0002090930938720703 sec
Compiling broadcast_in_dim for with global shapes and types [ShapedArray(float64[])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module conversion jit(broadcast_in_dim) in 0.0009779930114746094 sec
Finished XLA compilation of jit(broadcast_in_dim) in 0.04252910614013672 sec
Finished tracing + transforming <lambda> for pjit in 0.0002875

Finished tracing + transforming jit(squeeze) in 0.00018787384033203125 sec
Compiling squeeze for with global shapes and types [ShapedArray(uint32[1,2])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module conversion jit(squeeze) in 0.0011248588562011719 sec
Finished XLA compilation of jit(squeeze) in 0.010838747024536133 sec
Compiling _threefry_split_foldlike for with global shapes and types [ShapedArray(uint32[2])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module conversion jit(_threefry_split_foldlike) in 0.002376079559326172 sec
Finished XLA compilation of jit(_threefry_split_foldlike) in 0.05872821807861328 sec
Finished tracing + transforming _unstack for pjit in 0.0005159378051757812 sec
Compiling _unstack for with global shapes and types [ShapedArray(uint32[2,2])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module conversion jit(_unstack) in 0.0013303756713867188 sec
Finished XLA compilatio

### benchmark the sampler

In [7]:
from netket.sampler.metropolis import MetropolisLocal
from functools import partial

In [8]:
@partial(jax.jit, static_argnums=(1, 4))
def _sample_chain(sampler, machine, parameters, state, chain_length):
    state, samples = jax.lax.scan(
        lambda state, _: sampler.sample_next(machine, parameters, state),
        state,
        xs=None,
        length=chain_length,
    )

    return samples, state

In [9]:
x1 = jax.block_until_ready(_sample_chain(sa, ma, vs1.variables, sampler_state1, 128))

Finished tracing + transforming fn for pjit in 0.0002796649932861328 sec
Finished tracing + transforming real for pjit in 0.00018835067749023438 sec
Finished tracing + transforming right_shift for pjit in 0.00037550926208496094 sec
Finished tracing + transforming signbit for pjit in 0.0012929439544677734 sec
Finished tracing + transforming fn for pjit in 0.00032067298889160156 sec
Finished tracing + transforming fn for pjit in 0.0002770423889160156 sec
Finished tracing + transforming fn for pjit in 0.00031638145446777344 sec
Finished tracing + transforming fn for pjit in 0.0003108978271484375 sec
Finished tracing + transforming <lambda> for pjit in 0.00020432472229003906 sec
Finished tracing + transforming <lambda> for pjit in 0.00019884109497070312 sec
Finished tracing + transforming fn for pjit in 0.00025963783264160156 sec
Finished tracing + transforming <lambda> for pjit in 0.0003790855407714844 sec
Finished tracing + transforming _reduce_sum for pjit in 0.00034737586975097656 sec


In [10]:
x2 = jax.block_until_ready(_sample_chain(sa, ma, vs2.variables, sampler_state2, 128))

Compiling _sample_chain for with global shapes and types [ShapedArray(int64[], weak_type=True), ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(complex128[32]), ShapedArray(float64[512,32]), ShapedArray(uint32[2]), ShapedArray(int64[]), ShapedArray(int64[])]. Argument mapping: (GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({devices=[2,1]0,1}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated})).
Finished jaxpr to MLIR module conversion jit(_sample_chain) in 0.05135011672973633 sec
Finished XLA compilation of jit(_sample_chain) in 0.7835967540740967 sec
Finished tracing + transforming _multi_slice for pjit in 0.0003845691680908203 sec
Compiling _multi_slice for with global shapes and types [ShapedArray(complex128[256])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module conversion jit(_multi_slice) in 0.0012118816

- absolutely terrible scaling
- need to investigate

In [11]:
%timeit _ = jax.block_until_ready(_sample_chain(sa, ma, vs1.variables, sampler_state1, 128))

884 ms ± 16.6 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
%timeit _ = jax.block_until_ready(_sample_chain(sa, ma, vs2.variables, sampler_state2, 128))

633 ms ± 147 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
sap = netket.experimental.sampler.MetropolisSamplerPmap(hi, nk.sampler.rules.LocalRule(), n_chains=512)

In [14]:
sampler_statep = sap.init_state(ma, vs1.variables)

Finished tracing + transforming <lambda> for pjit in 0.00026869773864746094 sec
Finished tracing + transforming <lambda> for pjit in 0.0002560615539550781 sec
Finished tracing + transforming clip for pjit in 0.0019261837005615234 sec
Finished tracing + transforming <lambda> for pjit in 0.0002529621124267578 sec
Finished tracing + transforming <lambda> for pjit in 0.0002467632293701172 sec
Finished tracing + transforming clip for pjit in 0.0016894340515136719 sec
Finished tracing + transforming <lambda> for pjit in 0.0003037452697753906 sec
Finished tracing + transforming fn for pjit in 0.0002455711364746094 sec
Finished tracing + transforming fn for pjit in 0.0002434253692626953 sec
Finished tracing + transforming <lambda> for pjit in 0.00024366378784179688 sec
Finished tracing + transforming _randint for pjit in 0.010717630386352539 sec
Finished tracing + transforming fn for pjit in 0.00029540061950683594 sec
Finished tracing + transforming <lambda> for pjit in 0.0003056526184082031 s

In [15]:
xp = jax.block_until_ready(_sample_chain(sap, ma, vs1.variables, sampler_statep, 128))

Finished tracing + transforming <lambda> for pjit in 0.0002906322479248047 sec
Finished tracing + transforming <lambda> for pjit in 0.0002741813659667969 sec
Finished tracing + transforming fn for pjit in 0.00028061866760253906 sec
Finished tracing + transforming fn for pjit in 0.0002856254577636719 sec
Finished tracing + transforming _uniform for pjit in 0.00491642951965332 sec
Finished tracing + transforming _normal_real for pjit in 0.006336688995361328 sec
Finished tracing + transforming fn for pjit in 0.0003037452697753906 sec
Finished tracing + transforming fn for pjit in 0.0002582073211669922 sec
Finished tracing + transforming true_divide for pjit in 0.00027561187744140625 sec
Finished tracing + transforming _normal for pjit in 0.010421037673950195 sec
Finished tracing + transforming fn for pjit in 0.0003197193145751953 sec
Finished tracing + transforming <lambda> for pjit in 0.0003631114959716797 sec
Finished tracing + transforming <lambda> for pjit in 0.00026702880859375 sec
F

In [16]:
%timeit _ = jax.block_until_ready(_sample_chain(sap, ma, vs1.variables, sampler_statep, 128))

648 ms ± 204 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [17]:
x1[0].shape, x2[0].shape, xp[0].shape

((128, 512, 32), (128, 512, 32), (128, 512, 32))

In [18]:
x1[0].sharding, x2[0].sharding, xp[0].sharding

(SingleDeviceSharding(device=gpu(id=0)),
 PositionalSharding([[[{GPU 0}]
                      [{GPU 1}]]]),
 GSPMDSharding({replicated}))

In [19]:
x2[0].sharding.shape

(1, 2, 1)

In [20]:
x1 = jax.block_until_ready(vs1.sample())
x2 = jax.block_until_ready(vs2.sample())
x1 = jax.block_until_ready(vs1.sample())
x2 = jax.block_until_ready(vs2.sample())

Finished tracing + transforming _sample_chain for pjit in 0.02846813201904297 sec
Compiling _sample_chain for with global shapes and types [ShapedArray(int64[], weak_type=True), ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(complex128[32]), ShapedArray(float64[512,32]), ShapedArray(uint32[2]), ShapedArray(int64[], weak_type=True), ShapedArray(int64[], weak_type=True)]. Argument mapping: (GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated})).
Finished jaxpr to MLIR module conversion jit(_sample_chain) in 0.05246162414550781 sec
Finished XLA compilation of jit(_sample_chain) in 0.45906639099121094 sec
Compiling _sample_chain for with global shapes and types [ShapedArray(int64[], weak_type=True), ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(complex128[32])

In [21]:
%timeit jax.block_until_ready(vs1.sample())

112 ms ± 2.93 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [22]:
%timeit jax.block_until_ready(vs2.sample())

80.4 ms ± 3.27 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


### benchmark gradients (includes numba operator on the cpu)

In [23]:
eg1 = jax.block_until_ready(vs1.expect_and_grad(ha))
eg2 = jax.block_until_ready(vs2.expect_and_grad(ha))

Finished tracing + transforming jit(reshape) in 0.00020456314086914062 sec
Compiling reshape for with global shapes and types [ShapedArray(float64[16,512,32])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module conversion jit(reshape) in 0.0010704994201660156 sec
Finished XLA compilation of jit(reshape) in 0.011351346969604492 sec
Finished tracing + transforming atleast_2d for pjit in 0.0002086162567138672 sec
Finished tracing + transforming fn for pjit in 0.00026988983154296875 sec
Finished tracing + transforming real for pjit in 0.0001742839813232422 sec
Finished tracing + transforming right_shift for pjit in 0.0003571510314941406 sec
Finished tracing + transforming signbit for pjit in 0.0014960765838623047 sec
Finished tracing + transforming fn for pjit in 0.00030493736267089844 sec
Finished tracing + transforming fn for pjit in 0.0002560615539550781 sec
Finished tracing + transforming fn for pjit in 0.0003025531768798828 sec
Finished tracing + transfor

In [24]:
%timeit eg1 = jax.block_until_ready(vs1.expect_and_grad(ha))

104 ms ± 91.5 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [25]:
%timeit eg2 = jax.block_until_ready(vs2.expect_and_grad(ha))

70.5 ms ± 48.7 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


### benchmark SR

In [26]:
S1 = vs1.quantum_geometric_tensor(sr.qgt_constructor)
S1p = vs1.quantum_geometric_tensor(srp.qgt_constructor)
S2 = vs2.quantum_geometric_tensor(sr.qgt_constructor)
S2p = vs2.quantum_geometric_tensor(srp.qgt_constructor)

Finished tracing + transforming mat_vec_factory for pjit in 0.012855291366577148 sec
Compiling mat_vec_factory for with global shapes and types [ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(float64[16,512,32])]. Argument mapping: (GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated})).
Finished jaxpr to MLIR module conversion jit(mat_vec_factory) in 0.004731655120849609 sec
Finished XLA compilation of jit(mat_vec_factory) in 0.1360948085784912 sec
Finished tracing + transforming jacobian_default_mode for pjit in 0.00027942657470703125 sec
Finished tracing + transforming _reduce_sum for pjit in 0.00043010711669921875 sec
Finished tracing + transforming _mean for pjit in 0.0015876293182373047 sec
Finished tracing + transforming true_divide for pjit in 0.0003151893615722656 sec
Finished tracing + transforming <lambda> for pjit in 0.00027298927307128906 sec
Finished tracing + transforming true_divide for pjit in 0.00031948089599

In [27]:
jax.tree_util.tree_leaves(S1p.O)[1].sharding

SingleDeviceSharding(device=gpu(id=0))

In [28]:
jax.tree_util.tree_leaves(S2p.O)[1].sharding.shape

(1, 2, 1, 1)

In [29]:
jax.tree_util.tree_leaves(S1._mat_vec)[0].sharding

SingleDeviceSharding(device=gpu(id=0))

In [30]:
jax.tree_util.tree_leaves(S2._mat_vec)[0].sharding.shape

(1, 2, 1)

In [31]:
@jax.jit
def mv(S, v):
    return S@v

In [32]:
_ = jax.block_until_ready(mv(S1, vs1.parameters))
_ = jax.block_until_ready(mv(S1p, vs1.parameters))
_ = jax.block_until_ready(mv(S2, vs2.parameters))
_ = jax.block_until_ready(mv(S2p, vs2.parameters))

Finished tracing + transforming fn for pjit in 0.0003292560577392578 sec
Finished tracing + transforming _reduce_sum for pjit in 0.0004284381866455078 sec
Finished tracing + transforming _mean for pjit in 0.0014424324035644531 sec
Finished tracing + transforming true_divide for pjit in 0.00038886070251464844 sec
Finished tracing + transforming <lambda> for pjit in 0.0002770423889160156 sec
Finished tracing + transforming fn for pjit in 0.0003209114074707031 sec
Finished tracing + transforming fn for pjit in 0.0003151893615722656 sec
Finished tracing + transforming fn for pjit in 0.00031065940856933594 sec
Finished tracing + transforming onthefly_mat_treevec for pjit in 0.022493839263916016 sec
Finished tracing + transforming mv for pjit in 0.02408289909362793 sec
Compiling mv for with global shapes and types [ShapedArray(float64[], weak_type=True), ShapedArray(complex128[16,512,32]), ShapedArray(complex128[16,512,256]), ShapedArray(complex128[]), ShapedArray(complex128[16,512,256]), Sh

In [33]:
%timeit jax.block_until_ready(mv(S1, vs1.parameters))

4.66 ms ± 5.78 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [34]:
%timeit jax.block_until_ready(mv(S2, vs2.parameters))

8.02 ms ± 1.29 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [35]:
jvp_fn, = S1._mat_vec.args

In [36]:
from netket.optimizer.qgt.qgt_onthefly_logic import *

def _O_jvp(forward_fn, params, samples, v):
    _, res = jax.jvp(lambda p: forward_fn(p, samples), (params,), (v,))
    return res


def _O_vjp(forward_fn, params, samples, w):
    _, vjp_fun = jax.vjp(forward_fn, params, samples)
    res, _ = vjp_fun(w)
    return res

def _OH_w(forward_fn, params, samples, w):
    return tree_conj(_O_vjp(forward_fn, params, samples, w.conjugate()))


def _Odagger_DeltaO_v(forward_fn, params, samples, v):
    w = _O_jvp(forward_fn, params, samples, v)
    w = w * (1.0 / (samples.shape[0] * samples.shape[1] * mpi.n_nodes))
    #w_mean = w.sum(axis=(0,1), keepdims=True) / (samples.shape[0] * samples.shape[1] * mpi.n_nodes)
    #w_mean, _ = mpi.mpi_sum_jax(w_mean)
    #w = w - w_mean
    res = _OH_w(forward_fn, params, samples, w)
    return jax.tree_map(lambda x: mpi.mpi_sum_jax(x)[0], res)  # MPI




In [37]:
@partial(jax.jit, static_argnums=0)
def mv(forward_fn, params, samples, v, diag_shift):
    f = lambda p, x: jax.vmap(lambda x: forward_fn({'params': p},x))(x)
    res = _Odagger_DeltaO_v(f, params, samples, v)
    return tree_axpy(diag_shift, v, res)

In [38]:
y1 = jax.block_until_ready(mv(vs1._apply_fun, vs1.parameters, vs1.samples, vs1.parameters, 0.))
y2 = jax.block_until_ready(mv(vs2._apply_fun, vs2.parameters, vs2.samples, vs2.parameters, 0.))

Finished tracing + transforming mv for pjit in 0.03132200241088867 sec
Compiling mv for with global shapes and types [ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(float64[16,512,32]), ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(complex128[32]), ShapedArray(float64[], weak_type=True)]. Argument mapping: (GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated})).
Finished jaxpr to MLIR module conversion jit(mv) in 0.01532435417175293 sec
Finished XLA compilation of jit(mv) in 0.3617377281188965 sec
Compiling mv for with global shapes and types [ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(float64[16,512,32]), ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(complex128[32]), ShapedArray(float64[], weak_type=True)]. Argument mapping

In [39]:
%timeit _ = jax.block_until_ready(mv(vs1._apply_fun, vs1.parameters, vs1.samples, vs1.parameters, 0.))

6.08 ms ± 1.61 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [40]:
%timeit _ = jax.block_until_ready(mv(vs2._apply_fun, vs2.parameters, vs2.samples, vs2.parameters, 0.))

9.2 ms ± 89.8 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


- otf is much slower; figure out why

#### pytree

In [47]:
@jax.jit
def mv(S, v):
    return S@v

In [50]:
_  =jax.block_until_ready(mv(S1p, vs1.parameters))
_  =jax.block_until_ready(mv(S2p, vs2.parameters))

In [51]:
%timeit jax.block_until_ready(mv(S1p, vs1.parameters))

6.67 ms ± 9.83 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [52]:
%timeit jax.block_until_ready(mv(S2p, vs2.parameters))

4.28 ms ± 26.6 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


- pytree is faster

### vmc

In [53]:
gs1 = nk.VMC(ha, op, variational_state=vs1, preconditioner=srp)
gs2 = nk.VMC(ha, op, variational_state=vs2, preconditioner=srp)

In [54]:
gs1.run(100)

No output specified (out=[apath|nk.logging.JsonLogger(...)]).Running the optimization but not saving the output.


  0%|                                                                                                                                     | 0/100 [00:00<?, ?it/s]Finished tracing + transforming _matmul for pjit in 0.003981828689575195 sec
Finished tracing + transforming real for pjit in 0.00017595291137695312 sec
Finished tracing + transforming ravel for pjit in 0.00014328956604003906 sec
Finished tracing + transforming dot for pjit in 0.00034499168395996094 sec
Finished tracing + transforming vdot for pjit in 0.001779794692993164 sec
Finished tracing + transforming imag for pjit in 0.00016999244689941406 sec
Finished tracing + transforming fn for pjit in 0.0002474784851074219 sec
Finished tracing + transforming real for pjit in 0.00024008750915527344 sec
Finished tracing + transforming ravel for pjit in 0.00020051002502441406 sec
Finished tracing + transforming dot for pjit in 0.0003337860107421875 sec
Finished tracing + transforming vdot for pjit in 0.002198457717895508 sec
Finished 

()

In [55]:
gs2.run(100)

No output specified (out=[apath|nk.logging.JsonLogger(...)]).Running the optimization but not saving the output.


  0%|                                                                                                                                     | 0/100 [00:00<?, ?it/s]Compiling _solve for with global shapes and types [ShapedArray(float64[], weak_type=True), ShapedArray(complex128[16,512,256]), ShapedArray(complex128[16,512,32,256]), ShapedArray(complex128[16,512,32]), ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(complex128[32]), ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(complex128[32])]. Argument mapping: (GSPMDSharding({replicated}), GSPMDSharding({devices=[1,2,1]0,1}), GSPMDSharding({devices=[1,2,1,1]0,1}), GSPMDSharding({devices=[1,2,1]0,1}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated})).
Finished jaxpr to MLIR module conversion jit(_solve) in 0.0344090461730957 sec
Finished XLA compilation of jit(_s

()